# 03 · Adnotacja tweetów Trumpa (annotation): codebook → arkusze Excel → gold standard → walidacja instrumentu

Ten notebook prowadzi przez cały proces. Uruchamiasz go **kilka razy**, w miarę postępu prac:

| Kiedy | Co robi notebook | Co robicie Wy |
|---|---|---|
| 1. start | losuje próbkę i tworzy arkusze Excela (listy rozwijane) w `adnotacja/` | rozsyłacie pliki `1_pilotaz_<inicjały>.xlsx` |
| 2. po pilotażu | liczy zgodność na 50 postach, pokazuje rozbieżności | spotkanie, poprawka codebooka, potem pliki `2_glowna_<inicjały>.xlsx` |
| 3. po całości | kappa na nakładce, arkusz `rozstrzyganie.xlsx` | wpisujecie `FINAL_*` i uzasadnienie przy sporach |
| 4. po rozstrzygnięciu | gold standard, walidacja instrumentu, połączenie z danymi scalonymi | — |

**Zasada ślepej adnotacji:** arkusze nie zawierają reakcji rynku ani polubień. Klucz łączący posty z danymi (`klucz_probki.csv`) zostaje u osoby prowadzącej.

Wymaga: `02_merging.ipynb` (pliki `clean_out/merged/`, do połączenia etykiet z rynkiem w etapie G).

In [1]:
import os, sys, types, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60, "display.width", 220, "display.max_colwidth", 80)

PROJECT_DIR = None   # opcjonalnie ścieżka do folderu projektu (tego z cleaning_data/ i merging_data/); None = szukaj

def find_project():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "cleaning_data" / "__init__.py").exists() and (d / "merging_data" / "__init__.py").exists():
            return d
    return None

root = Path(PROJECT_DIR) if PROJECT_DIR else find_project()
if root is None:
    raise SystemExit(f"Nie znaleziono folderu projektu z cleaning_data/ i merging_data/ (katalog roboczy: {Path.cwd()}). "
                     "Ustaw PROJECT_DIR.")
os.chdir(root)                                   # ścieżki danych (clean_out/, data/, ...) są względem folderu projektu
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
for m in [m for m in sys.modules if m.split(".")[0] in ("cleaning_data", "merging_data", "annotation", "explore_pkg")]:
    del sys.modules[m]                           # świeże wczytanie po podmianie plików (bez restartu kernela)
import cleaning_data, merging_data
PACKAGES = [cleaning_data, merging_data]
import annotation
PACKAGES.append(annotation)
pkg = types.SimpleNamespace(**{k: getattr(p, k) for p in PACKAGES for k in p.__all__})
A, show, cfg = annotation.annotation, pkg.show, pkg.Config()

ANNOT    = Path("annotation_files")           # folder z arkuszami (poza clean_out, bo scalanie go nadpisuje)
MERGED   = Path("clean_out/merged")
N_SAMPLE = 400          # wielkość próbki (300-500)
PILOT, OVERLAP = 50, 120
RESAMPLE = False        # True = wylosuj od nowa (zablokowane, gdy ktoś już coś wpisał)
DRY_RUN  = False        # True = próba generalna: arkusze wypełnia automat z szumem (wyniki NIE do pracy)
print("Projekt:", root, "| codebook", A.CODEBOOK_VERSION, "| oznaczający:", A.ANNOTATORS)

Projekt: C:\Users\pauko\Desktop\Studia\Magisterka\2 semestr\Analiza mediów cyfrowych\Laby\tariffs-vs-social-media | codebook 0.1.2 | oznaczający: ('BB', 'IK', 'MG', 'PK')


## A. Próbka i arkusze do wypełnienia
Losowanie warstwowe (miesiąc × trafienie słownika), połowa próbki to posty, które słownik uznał za cła, a połowa pozostałe.
Dzięki temu są przypadki pozytywne do oznaczenia i da się zmierzyć, ile tweetów o cłach słownik **pomija**.
Każdy post ma wagę (odwrotność prawdopodobieństwa wylosowania), żeby miary słownika przenieść na cały korpus.

Gdy plan już istnieje, jest **wczytywany, a nie losowany od nowa**: po rozesłaniu arkuszy próbka nie może się zmienić. Brakujące arkusze .xlsx są wtedy tworzone z tego samego planu, a puste arkusze .csv z poprzedniej wersji usuwane.

In [2]:
tr = pkg.CleanLoader(cfg).trump()
key = ANNOT / "klucz_probki.csv"
filled_any = any(A._has_labels(p) for p in list(ANNOT.glob("[12]_*_*.xlsx")) + list(ANNOT.glob("[12]_*_*.csv")))
if key.exists() and not RESAMPLE:
    plan = A.load_plan(ANNOT)
    print(f"Wczytano istniejący plan: {len(plan)} postów ({key}).")
    for m in pkg.write_sheets(plan, ANNOT):     # brakujące arkusze .xlsx; wypełnionych nie nadpisuje
        print(m)
elif RESAMPLE and filled_any:
    raise SystemExit("RESAMPLE=True, ale w arkuszach są już wpisy. Nie losuję od nowa (zmieniłoby to próbkę). Ustaw RESAMPLE=False.")
else:
    sampler = pkg.AnnotationSampler(cfg, n=N_SAMPLE, pilot=PILOT, overlap=OVERLAP)
    plan = sampler.assign(sampler.draw(tr))
    for m in pkg.write_sheets(plan, ANNOT):
        print(m)

show(pd.crosstab(plan["etap"], plan["hit"].map({True: "słownik: tak", False: "słownik: nie"}), margins=True, margins_name="razem"),
     "próbka: etap × trafienie słownika")
load = pd.Series([a for s in plan["oznaczajacy"] for a in s.split("|")]).value_counts().rename("postów do oznaczenia")
show(load.to_frame(), "obciążenie na osobę")

zapisano: annotation_files\1_pilotaz_BB.xlsx (50 wierszy)
zapisano: annotation_files\2_glowna_BB.xlsx (118 wierszy)
zapisano: annotation_files\1_pilotaz_IK.xlsx (50 wierszy)
zapisano: annotation_files\2_glowna_IK.xlsx (118 wierszy)
zapisano: annotation_files\1_pilotaz_MG.xlsx (50 wierszy)
zapisano: annotation_files\2_glowna_MG.xlsx (117 wierszy)
zapisano: annotation_files\1_pilotaz_PK.xlsx (50 wierszy)
zapisano: annotation_files\2_glowna_PK.xlsx (117 wierszy)
zapisano: annotation_files\klucz_probki.csv (400 wierszy)
zapisano: annotation_files\INSTRUKCJA.md

=== próbka: etap × trafienie słownika ===


hit,słownik: nie,słownik: tak,razem
etap,,,
nakladka,61,59,120
pilotaz,22,28,50
pojedyncze,117,113,230
razem,200,200,400



=== obciążenie na osobę ===


,postów do oznaczenia
BB,168
IK,168
MG,167
PK,167


**Co teraz:** każdy dostaje swój plik `1_pilotaz_<inicjały>.xlsx` (karta *Instrukcja* w środku). Wypełnia się tylko żółte kolumny, z list rozwijanych. Po pilotażu uruchom etapy B i C,
omówcie rozbieżności i dopiero wtedy wypełniajcie `2_glowna_<inicjały>.xlsx`. Wypełnione pliki wracają do folderu `adnotacja/` pod tą samą nazwą.

## B. Wczytanie arkuszy i kontrola wypełnienia
Wartości są sprowadzane do listy z codebooka (wielkość liter, polskie znaki, skróty). Wszystko inne trafia do raportu problemów.

In [3]:
SRC = ANNOT / "_symulacja" if DRY_RUN else ANNOT
if DRY_RUN:
    print("PRÓBA GENERALNA: arkusze wypełnia automat, wyniki NIE są danymi do pracy.")
    for m in A.simulate_filled(ANNOT, SRC, plan): print(m)
L, problems = pkg.load_sheets(SRC, plan)
READY = len(L) > 0
if not READY:
    print("Brak wypełnionych arkuszy. Po wypełnieniu pilotażu uruchom notebook ponownie od tej komórki.")
else:
    show(pkg.progress(L, plan), "postęp")
    if len(problems):
        show(problems.groupby(["plik", "problem"]).size().rename("ile").to_frame(), "problemy w arkuszach (popraw i wczytaj ponownie)")
        show(problems.head(20), "pierwsze problemy")
    else:
        print("Brak problemów w arkuszach.")

Brak wypełnionych arkuszy. Po wypełnieniu pilotażu uruchom notebook ponownie od tej komórki.


## C. Zgodność między oznaczającymi
- **Pilotaż** (4 osoby, 50 postów): alfa Krippendorffa. Służy poprawie codebooka i **nie wchodzi** do końcowej oceny.
- **Nakładka** (pary, ~120 postów): kappa Cohena dla każdej pary i alfa. **Cel ≥ 0,70** dla cla i kierunku.
- `kierunek | cla=tak` liczony tylko na postach, które obie osoby uznały za cła (to właściwa miara dla tej etykiety).

In [4]:
if not READY:
    print("Brak danych: etap B.")
else:
    for stage in ("pilotaz", "nakladka"):
        ag = pkg.agreement(L, plan, stage)
        if ag.empty:
            print(f"{stage}: jeszcze brak oznaczeń od co najmniej dwóch osób.")
            continue
        show(ag[ag["para"] == "RAZEM"].drop(columns="para"), f"zgodność: {stage}")
        show(ag[ag["para"] != "RAZEM"].pivot_table(index="para", columns="etykieta", values="kappa"), f"kappa per para: {stage}")
    cm = A.confusion_between(L, plan, "kierunek", "pilotaz")
    if len(cm):
        show(cm, "pilotaż: gdzie mylimy kierunek (wiersz = osoba A, kolumna = osoba B)")

Brak danych: etap B.


## D. Rozstrzyganie i gold standard
1. Komórka zapisuje `rozstrzyganie.xlsx` ze spornymi postami. Tam, gdzie jest zgoda albo większość, `FINAL_*` jest już wpisane.
2. Na spotkaniu uzupełniacie puste `FINAL_*` i `uzasadnienie`. Plik nie jest nadpisywany, gdy ma już wpisy.
3. Ponowne uruchomienie buduje `gold_standard.csv` i dzieli go na część **dev** (40%, dobór promptu/modelu) i **test** (60%, ocena końcowa, bez podglądania).

In [5]:
if not READY:
    print("Brak danych: etap B.")
else:
    ADJ = SRC / "rozstrzyganie.xlsx"
    print(pkg.write_adjudication(L, plan, ADJ))
    gold, open_items = pkg.build_gold(L, plan, ADJ)
    gold.to_csv(SRC / "gold_standard.csv", sep=A.SEP, index=False, encoding=A.ENC)   # do skryptów
    gold.assign(time_utc=gold["time_utc"].astype(str)).to_excel(SRC / "gold_standard.xlsx", index=False)  # do przeglądania
    print(f"gold standard: {len(gold)} postów -> {SRC / 'gold_standard.xlsx'} (+ .csv) | nierozstrzygnięte: {len(open_items)}")
    if len(open_items):
        show(open_items[["item_id"] + [c for c in open_items if c.endswith("_oceny")]].head(10), "do rozstrzygnięcia (FINAL_* w rozstrzyganie.xlsx)")
    show(gold["zrodlo_etykiety"].value_counts().to_frame("posty"), "skąd etykieta")
    show(A.label_distribution(gold), "rozkład etykiet i podział dev/test")

Brak danych: etap B.


## E. Walidacja instrumentu
Instrumentem są dwie rzeczy: **codebook z adnotatorami** (rzetelność: zgodność z etapu C) i **słownik ceł**, który w notebooku 02 wybiera tweety-zdarzenia (trafność: porównanie z gold).
- **Precyzja słownika:** jaki odsetek tweetów uznanych przez słownik za cła naprawdę dotyczy ceł.
- **Czułość:** jaki odsetek tweetów o cłach słownik w ogóle łapie.
- Miary **ważone** przenoszą wynik z próbki (nadreprezentacja trafień) na cały korpus 2018-2019.

In [6]:
if not READY:
    print("Brak danych: etap B.")
else:
    val, errs = pkg.dictionary_validity(gold, tr)
    show(val, "trafność słownika ceł względem gold standardu")
    show(errs.groupby("błąd").size().rename("posty").to_frame(), "pomyłki słownika")
    show(errs.head(10), "przykłady pomyłek (do poprawy słownika)")
    ag = pkg.agreement(L, plan, "nakladka")
    if len(ag):
        show(ag[ag["para"] == "RAZEM"][["etykieta", "n", "kappa", "alfa_Krippendorffa", "ocena", "cel_≥_0,70"]], "rzetelność (nakładka)")

Brak danych: etap B.


## F. Ocena klasyfikatora: punkty odniesienia
Zanim użyjecie LLM lub zero-shot do oznaczenia **wszystkich** tweetów, musi on pokonać dwa proste punkty odniesienia na części **test**:
klasę najczęstszą i reguły słownikowe z codebooka. Komórka zapisuje też prompty (z przykładami z części dev) do `prompty_test.csv`.
Odpowiedzi modelu wczytujecie funkcją `A.parse_llm` i oceniacie tak samo: `pkg.evaluate(gold, predykcje)`.

In [7]:
if not READY:
    print("Brak danych: etap B.")
else:
    test = gold[gold["split"] == "test"]
    for name, pred in (("klasa najczęstsza", A.majority_baseline(gold)), ("reguły słownikowe", pkg.KeywordBaseline().predict(gold))):
        r = pkg.evaluate(gold, pred, split="test")
        show(A.evaluation_table(r), f"{name}: część test (n = {len(test)})")
    r = pkg.evaluate(gold, pkg.KeywordBaseline().predict(gold), split="test")
    if "kierunek" in r:
        show(r["kierunek"]["confusion"], "reguły słownikowe: macierz pomyłek kierunku (test)")
    prompts = A.build_prompts(test, gold[gold["split"] == "dev"], k_per_class=1)
    prompts.to_csv(SRC / "prompty_test.csv", sep=A.SEP, index=False, encoding=A.ENC)
    print("prompty:", SRC / "prompty_test.csv", "| przykład:\n", prompts["prompt"].iat[0][:700])

Brak danych: etap B.


## G. Połączenie etykiet z danymi scalonymi
- **Post → zdarzenie:** `event_id = "tw_" + post_id` w `events.csv`, `event_table.csv`, `market_long.csv`.
- **Post → epizod:** post należy do epizodu, który zaczął się najpóźniej przed nim (ta sama reguła 6 h co w notebooku 02). Etykieta epizodu = udział eskalacji i deeskalacji, kierunek pierwszego posta.
- **Post → dzień:** `eff_day` (od 16:00 NY następny dzień), czyli klucz paneli `daily`/`trading`. `add_label_counts` dodaje dzienne liczniki per etykieta.
- Posty z gold, których słownik nie złapał, **nie są** zdarzeniami w `merged`. Służą walidacji, a do analizy rynku wejdą dopiero po oznaczeniu całego korpusu (sekcja H).

Na gold standardzie (kilkaset postów, z czego część przy zamkniętym rynku) poniższe tabele są **opisowe**, nie do wniosków.

In [8]:
if not READY:
    print("Brak danych: etap B.")
elif not (MERGED / "events.csv").exists():
    print("Brak clean_out/merged: uruchom 02_merging.ipynb.")
else:
    link = pkg.link_to_merged(gold, MERGED)
    show(link["coverage"], "pokrycie")
    show(link["episodes"].head(), "etykiety epizodów")
    if "market" in link:
        show(pkg.market_by_label(link["market"], "kierunek", "S&P 500", "post").round(2), "S&P 500: nadwyżka |zwrotu| [pb] wg kierunku (posty z gold)")
    daily = pd.read_csv(MERGED / "daily.csv", index_col="day", parse_dates=["day"])
    daily_lab = pkg.add_label_counts(daily, gold[gold["cla"] == "tak"])
    show(daily_lab.filter(like="tw_kierunek").sum().to_frame("posty"), "dzienne liczniki z etykiet (gold)")

Brak danych: etap B.


## H. Po klasyfikatorze: cały korpus
Gdy klasyfikator przejdzie ocenę na części test, oznaczcie nim wszystkie oryginalne tweety 2018-2019 i przebudujcie scalenie,
wybierając zdarzenia z etykiety `cla` zamiast słownika. Kierunek trafi wtedy do każdego zdarzenia-posta i do paneli:

```python
pred = pd.read_csv("adnotacja/predykcje_korpus.csv", sep=";", dtype={"post_id": str})   # post_id, cla, kierunek
tr2 = tr.merge(pred, on="post_id", how="left")
tr2["cla_tak"] = tr2["cla"].eq("tak")
res2 = pkg.build_merged(cfg, trump=tr2, relevant_col="cla_tak", save=False)   # zdarzenia = posty z cla=tak
link2 = pkg.link_to_merged(pred, ...)                                         # etykiety do market_long / epizodów
daily2 = pkg.add_label_counts(res2["daily"], tr2[tr2["cla_tak"]])           # liczniki ESKALACJA/DEESKALACJA/NEUTRALNE
trading2 = pkg.to_trading_days(daily2)                                       # panel do LP i PCMCI+
```
`save=False`, żeby nie nadpisać wyników ze słownika; zapisz `res2` do osobnego folderu, jeśli chcecie porównać oba warianty.